In [ ]:
### SD-3-5-Medium ###
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. 定义数据
method_list = [
    "SD-3.5-M",
    "FlowGRPO",
    "GRPO-Guard",
    "DiffusionNFT",
    "Ours",
    "FlowGRPO+Ours",
]

human_preference_specific_dict = {
    "SD-3.5-M": { "UniRwd": 3.30, "HPSv3": 10.03 },
    "FlowGRPO": { "UniRwd": 3.59, "HPSv3": 12.63 },
    "GRPO-Guard": { "UniRwd": 3.64, "HPSv3": 12.51 },
    "DiffusionNFT": { "UniRwd": 3.68, "HPSv3": 12.47 },
    "Ours": { "UniRwd": 3.46, "HPSv3": 12.77 },
    "FlowGRPO+Ours": { "UniRwd": 3.60, "HPSv3": 13.30 }, # [新增] 数据
}

oneig_benchmark_dict = {
    "SD-3.5-M": 0.357,
    "FlowGRPO": 0.260,
    "GRPO-Guard": 0.241,
    "DiffusionNFT": 0.175,
    "Ours": 0.305,
    "FlowGRPO+Ours": 0.273, # [新增] 数据
}

# 2. 归一化处理
uni_rwd_values = [human_preference_specific_dict[m]["UniRwd"] for m in method_list]
hpsv3_values = [human_preference_specific_dict[m]["HPSv3"] for m in method_list]

uni_rwd_min, uni_rwd_max = min(uni_rwd_values), max(uni_rwd_values)
hpsv3_min, hpsv3_max = min(hpsv3_values), max(hpsv3_values)

def normalize(value, min_val, max_val):
    if max_val == min_val: return 0.0
    return (value - min_val) / (max_val - min_val)

weight_uni_rwd = 0.5
weight_hpsv3 = 0.5

human_preference_dict = {}
for m in method_list:
    uni_rwd_norm = normalize(human_preference_specific_dict[m]["UniRwd"], uni_rwd_min, uni_rwd_max)
    hpsv3_norm = normalize(human_preference_specific_dict[m]["HPSv3"], hpsv3_min, hpsv3_max)
    weighted_avg = weight_uni_rwd * uni_rwd_norm + weight_hpsv3 * hpsv3_norm
    human_preference_dict[m] = weighted_avg

# 3. 绘图设置
plt.rcParams.update({
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,
})

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)

# 0) 你想要的绘制/图例顺序（注意：你的 "Ours" 对应这里的 "AFPO (HPDv3)"）
plot_order = [
    "SD-3.5-M",
    "FlowGRPO",
    "GRPO-Guard",
    "DiffusionNFT",
    "Ours",     
    "FlowGRPO+Ours"
]

# 1) 用“方法名 -> 样式”固定映射，保证颜色/形状永远不随顺序变化
style_map = {
    "SD-3.5-M":        {"marker": "o", "color": "#A4C6E2"},
    "FlowGRPO":         {"marker": "D", "color": "#AFD498"}, 
    "GRPO-Guard":       {"marker": "P", "color": "#AE9DC7"},
    "DiffusionNFT":     {"marker": "X", "color": "#e7dbd3"},
    "Ours":     {"marker": "s", "color": "#E4A39D"},
    "FlowGRPO+Ours":  {"marker": "h", "color": "#D86983"},
}

# 2) 后面所有用到 method_list 的地方，都换成 plot_order
#    （归一化也建议用 plot_order，保证 min/max 与展示方法一致）
method_list = plot_order

# ---------- 归一化（保持你原逻辑） ----------
uni_rwd_values = [human_preference_specific_dict[m]["UniRwd"] for m in method_list]
hpsv3_values   = [human_preference_specific_dict[m]["HPSv3"] for m in method_list]

uni_rwd_min, uni_rwd_max = min(uni_rwd_values), max(uni_rwd_values)
hpsv3_min, hpsv3_max     = min(hpsv3_values), max(hpsv3_values)

def normalize(value, min_val, max_val):
    if max_val == min_val: 
        return 0.0
    return (value - min_val) / (max_val - min_val)

weight_uni_rwd = 0.5
weight_hpsv3 = 0.5

human_preference_dict = {}
for m in method_list:
    uni_rwd_norm = normalize(human_preference_specific_dict[m]["UniRwd"], uni_rwd_min, uni_rwd_max)
    hpsv3_norm   = normalize(human_preference_specific_dict[m]["HPSv3"], hpsv3_min, hpsv3_max)
    human_preference_dict[m] = weight_uni_rwd * uni_rwd_norm + weight_hpsv3 * hpsv3_norm

# ---------- Plot points（用 style_map，而不是按 i 索引取 marker/color） ----------
base_size = 280
for m in method_list:
    x = human_preference_dict[m]
    y = oneig_benchmark_dict[m]
    lw = 1.0

    mk = style_map[m]["marker"]
    c  = style_map[m]["color"]

    # 保留你原来的大小/zorder逻辑
    if m == "FlowGRPO+Ours":
        s = base_size * 1.25
        zorder = 11
        alpha = 1.0
    else:
        zorder = 3
        alpha = 1.0
        s = base_size
        if m == "FlowGRPO":
            s = base_size * 90/160
        elif m == "Ours":
            s = base_size * 0.75

    
    ax.scatter(
        x, y,
        s=s, marker=mk, facecolors=c, edgecolors="black",
        linewidths=lw, zorder=zorder, label=m, alpha=alpha
    )
    
    
# 4. 标签布局
label_offsets = {
    "SD-3.5-M": (10, 0),
    "FlowGRPO": (-5, 9),            
    "GRPO-Guard": (0, -15),
    "DiffusionNFT": (0, -15),
    "Ours": (-10, 0),
    "FlowGRPO+Ours": (0, 20) 
}

fontsize_label = 11
for m in method_list:
    x = human_preference_dict[m]
    y = oneig_benchmark_dict[m]
    dx, dy = label_offsets.get(m, (5, 5))
    
    if dx < 0: ha = 'right'
    elif dx > 0: ha = 'left'
    else: ha = 'center'
    
    font_weight = "normal"
    
    if m == "FlowGRPO+Ours":
        display_name = r"FlowGRPO" + "\n" + r"$\mathbf{+Ours}$"
    elif "Ours" in m:
        display_name = m.replace("Ours", r"$\mathbf{Ours}$")
    else:
        display_name = m
        
    ax.annotate(
        display_name, (x, y),
        xytext=(dx, dy), textcoords="offset points",
        fontsize=fontsize_label, color="#333333", fontweight=font_weight,
        ha=ha, va='center', zorder=20
    )

# Axes settings
ax.set_xlabel(r"Human Preference$\uparrow$")
ax.set_ylabel(r"Stylization$\uparrow$")
ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.3, zorder=0)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# Ticks
ax.set_xlim(-0.1, 1.1)
ax.set_xticks(np.arange(0.0, 1.1, 0.2))
ax.set_ylim(0.14, 0.41)
ax.set_yticks(np.arange(0.15, 0.41, 0.05))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.2f}"))

# Legend
legend = ax.legend(
    loc="lower left", frameon=True, fancybox=True, edgecolor="gray",
    framealpha=0.3, borderpad=0.3, labelspacing=0.6,
    handletextpad=0.3, borderaxespad=0.8
)

# Legend marker sizing
# 注意：这里改用了 legendHandles 以兼容新版 matplotlib
for handle in legend.legend_handles:
    lbl = handle.get_label()
    if lbl == "FlowGRPO+Ours": # [新增] 图例大小
        handle.set_sizes([120*1.25])
    elif lbl == "Ours":
        handle.set_sizes([80.0/100*120]) 
    elif lbl == "FlowGRPO":
        handle.set_sizes([60/100*120]) 
    else:
        handle.set_sizes([100.0/100*120]) 

fig.tight_layout()
fig.savefig("human_preference-vs-oneig_style-v3.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
### SD-1.5 ###
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. 定义数据
method_list = [
    "SD-1.5",
    "Diffusion-KTO",
    "InPO",
    "SPO",
    "GradSPO",
    "Diffusion-DPO",
    "AFPO (HPDv3)"
]

human_preference_specific_dict = {
    "SD-1.5": { "UniRwd": 2.92, "HPSv3": 5.98 },
    "Diffusion-KTO": { "UniRwd": 3.15, "HPSv3": 7.77 },
    "InPO": { "UniRwd": 3.22, "HPSv3": 7.88 },
    "SPO": { "UniRwd": 2.98, "HPSv3": 7.31 },
    "GradSPO": { "UniRwd": 3.04, "HPSv3": 7.45 },
    "Diffusion-DPO": { "UniRwd": 3.03, "HPSv3": 6.80 },
    "AFPO (HPDv3)": { "UniRwd": 3.06, "HPSv3": 7.33 }
}

deqa_dict = {
    "SD-1.5": 3.70,
    "Diffusion-KTO": 3.59,
    "InPO": 3.64,
    "SPO": 3.40,
    "GradSPO": 2.92,
    "Diffusion-DPO": 3.78,
    "AFPO (HPDv3)": 3.96
}